# Analyse von *Instagram*-Daten, die mit `zeeschuimer` gesammelt wurden

In diesem Notebook schauen wir uns exemplarisch an, wie man automatisierte Textanalysen und Netzwerkanalysen mit *Instagram*-Daten durchführen lann, die mit dem Tool [`zeeschuimer`](https://github.com/digitalmethodsinitiative/zeeschuimer) gesammelt wurden.

Ausgangspunkt sind `.csv`-Dateien, die entweder mit [4CAT](https://4cat.nl/) oder dem im [Repository zur Exploration von *Instagram*-Daten](https://github.com/jobreu/insta-explore) inkludierten Parser für die mit `zeeschuimer` erstellten `.njdson`-Dateien.

## Datenimport

Zur Erinnerung: Die verwendete `.csv`-Datei muss über den Upload-Button (nach oben zeigender Pfeil) im File Explorer auf der linken Seite der Jupyter-Lab-Analyseumgebung hochgeladen werden.

In [ ]:
library(readr)

In [ ]:
insta <- read_csv("INSERT_FILE_NAME_HERE") # Namen der entsprechenden Datei (inkl. Dateiendung) einfügen

In [ ]:
names(insta)

In [ ]:
library(dplyr)

In [ ]:
glimpse(insta)

Wer sind die häufigsten Autor:innen im Datensatz?

In [ ]:
insta1 %>% 
  count(author) %>% 
  arrange(desc(n)) %>% 
  head(20)

## Textanalyse

In [ ]:
library(quanteda)

### Erstellung eines Corpus

In [ ]:
insta_corpus <- insta2 %>% 
  select(id, timestamp, unix_timestamp,
         url, body, 
         author, author_fullname,
         hashtags, usertags,
         num_likes, num_comments,
         location_name) %>% 
  corpus(docid_field = "id",
         text_field = "body")

insta_corpus

### Tokenisierung & Entfernung von Stop Words

In [ ]:
tokens_insta <- tokens(insta_corpus,
                       remove_punct = TRUE,
                       remove_symbols = TRUE,
                       remove_numbers = TRUE,
                       remove_url = TRUE)

In [ ]:
tokens_insta <- tokens_remove(tokens_insta,
                              stopwords("de"))

In [ ]:
tokens_insta

### Document-Feature-Matrix (DFM) erstellen

In [ ]:
dfm_insta <- dfm(tokens_insta)

dfm_insta

### Textdaten explorieren

In [ ]:
library(quanteda.textstats)

#### Worthäufigkeiten

In [ ]:
dfm_insta %>%
  dfm_remove(pattern = c("@*", "#*")) %>% # ohne User Tags und Hashtags
  textstat_frequency(n = 20)

#### Top Hashtags

In [ ]:
dfm_tag <- dfm_select(dfm_insta, pattern = "#*")
toptag <- names(topfeatures(dfm_tag, 50)) # 50 häufigste Hashtags
head(toptag, 10) # 10 häufigste Hashtags

#### Top User Tags

In [ ]:
dfm_users <- dfm_select(dfm_insta, pattern = "@*")
topuser <- names(topfeatures(dfm_users, 50)) # 50 häufigste User Tags
head(topuser, 10) # 10 häufigste User Tags

## Visualisierung

In [ ]:
library(quanteda.textplots)

#### Wortwolke

In [ ]:
dfm_insta %>% 
  dfm_remove(pattern = c("@*", "#*")) %>% # ohne User Mentions & Hashtags
  dfm_trim(min_termfreq = 10) %>% # Wörter müssen mind. 10x vorkommen
  textplot_wordcloud()

#### Plot zu Worthäufigkeiten

In [ ]:
tstat_freq <- dfm_insta %>% 
  dfm_remove(pattern = c("@*", "#*")) %>%
  textstat_frequency(n = 20) # Wörter müssen mind. 20x vorkommen

ggplot(tstat_freq, aes(x = frequency, y = reorder(feature, frequency))) +
  geom_col() + 
  labs(x = "Frequency", y = "Feature") +
  scale_x_continuous(expand = expansion(mult = c(0, 0.05)))


## Netzwerkanalyse

Die für die Netzwerkanalyse benötigten Daten können wir sowohl aus den Textdaten (mithilfe von Funktionen aus den `quanteda`-Paketen) als auch der ursprünglichen `.csv`-Datei generieren.

### Textdaten als Ausgangsbasis

#### Hashtag-Netzwerk

In [ ]:
fcmat_tag <- fcm(dfm_tag)
head(fcmat_tag)

In [ ]:
fcmat_topgat <- fcm_select(fcmat_tag, pattern = toptag)
textplot_network(fcmat_topgat, 
                 min_freq = 0.1,
                 edge_alpha = 0.8,
                 edge_size = 5)

#### User-Mention-Netzwerk

In [ ]:
fcmat_users <- fcm(dfm_users)
head(fcmat_users)

In [ ]:
fcmat_users <- fcm_select(fcmat_users, pattern = topuser)
textplot_network(fcmat_users,
                 min_freq = 0.1,
                 edge_color = "orange",
                 edge_alpha = 0.8,
                 edge_size = 5)

### `.csv`-Datei als Ausgangsbasis

#### Coauthor-Netzwerk

*Instagram* bietet die Option, dass Posts von mehreren Accounts gemeinsam veröffentlicht werden (Coauthors). Dies lässt sich auch als ungerichtetes Netzwerk darstellen. Die Anzahl der "Coauthored Posts" können wir als Gewicht (weight) für die Kanten (edges) nutzen.

**NB**: Die Erstellung eines Coauthor-Netzwerks ist nur mit über *4CAT* erstellten `.csv`-Dateien möglich, da der alternative Parser die entsprechende Variable nicht exportiert.

In [ ]:
library(igraph)

In [ ]:
coauthors <- insta1 %>%
  filter(!is.na(coauthors)) %>%
  separate_rows(coauthors, sep = ",") %>%
  mutate(coauthors = str_trim(coauthors)) %>%
  select(source = author, target = coauthors) %>% 
  count(source, target, name = "weight") %>% 
  filter(weight >= 2)

g1 = graph_from_data_frame(coauthors, 
                           directed = FALSE)

In [ ]:
plot(g1, 
     edge.width = E(g1)$weight * 2,
     edge.arrow.size = 0.5,
     edge.label = E(g1)$weight,
     main = "Weighted Coauthor Network")

#### User-Tag-Netzwerk

User Tags lassen sich als gerichtetes Netzwerk darstellen. Die Anzahl der Tags können wir als Gewicht (weight) für die Kanten (edges) nutzen.

In [ ]:
tags <- insta1 %>%
  filter(!is.na(usertags)) %>%
  separate_rows(usertags, sep = ",") %>%
  mutate(tags = str_trim(usertags)) %>%
  select(source = author, target = tags) %>% 
  count(source, target, name = "weight") %>% 
  filter(weight >= 2)

g2 = graph_from_data_frame(tags, directed = TRUE)

In [ ]:
plot(g2, 
     edge.width = E(g2)$weight * 2,
     edge.arrow.size = 0.5,
     edge.label = E(g2)$weight,
     main = "Weighted User Tag Network")

### Netzwerk-Indices

In [ ]:
mean_distance(g1, directed = FALSE)

In [ ]:
edge_density(g1)

In [ ]:
print(closeness(g1, normalized = T))

In [ ]:
print(degree(g2, normalized = T, mode="in"))
print(degree(g2, normalized = T, mode="out"))